In [ ]:
import cv2
import numpy as np
import time
from PIL import Image
from IPython.display import display

# Parametri – îi poți ajusta dacă vrei altă rezoluție / prag
FRAME_W = 640
FRAME_H = 480
NUM_FRAMES = 100      # câte cadre măsurăm
THRESH_VAL = 60       # prag pentru threshold
GAUSS_KERNEL = 5      # dim. filtru Gaussian (impar: 3,5,7,...)
GAUSS_SIGMA = 1.0     # sigma Gaussian, aproximativ ca în IP

# Inițializăm camera USB (indexul poate fi 0 sau 1, în funcție de sistem)
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Nu s-a putut deschide camera (index 0).")

# Setăm rezoluția (dacă driverul permite)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_H)

total_end_to_end = 0.0   # captură + procesare
total_proc_only = 0.0    # doar procesare
processed = 0
last_frame = None

for i in range(NUM_FRAMES):
    t0 = time.time()
    ret, frame = cap.read()
    if not ret:
        print(f"Eroare la citirea cadrului {i}.")
        break

    # Asigurare dimensiune (în caz că camera nu respectă setările)
    frame = cv2.resize(frame, (FRAME_W, FRAME_H))

    t1 = time.time()

    # === PIPELINE: BGR -> GRAY -> THRESH -> GAUSS -> SOBEL -> BGR ===
    proc_start = time.time()

    # 1. BGR -> GRAY
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 2. THRESH
    _, thr = cv2.threshold(gray, THRESH_VAL, 255, cv2.THRESH_BINARY)

    # 3. GAUSSIAN BLUR
    gauss = cv2.GaussianBlur(thr, (GAUSS_KERNEL, GAUSS_KERNEL), GAUSS_SIGMA)

    # 4. SOBEL (X+Y)
    sobelx = cv2.Sobel(gauss, cv2.CV_16S, 1, 0, ksize=3)
    sobely = cv2.Sobel(gauss, cv2.CV_16S, 0, 1, ksize=3)
    absx = cv2.convertScaleAbs(sobelx)
    absy = cv2.convertScaleAbs(sobely)
    sobel = cv2.addWeighted(absx, 0.5, absy, 0.5, 0)

    # 5. GRAY -> BGR (ieșire similară cu IP-ul)
    out_bgr = cv2.cvtColor(sobel, cv2.COLOR_GRAY2BGR)

    proc_end = time.time()

    # Timpuri
    total_end_to_end += (proc_end - t0)       # captură + procesare
    total_proc_only += (proc_end - proc_start)  # doar procesare
    processed += 1
    last_frame = out_bgr

cap.release()

if processed > 0:
    avg_end_ms = (total_end_to_end / processed) * 1000.0
    avg_proc_ms = (total_proc_only / processed) * 1000.0
    fps_end = processed / total_end_to_end
    fps_proc = processed / total_proc_only

    print(f"Cadre procesate efectiv:                 {processed}")
    print(f"Timp mediu end-to-end (captură+proc):   {avg_end_ms:.2f} ms")
    print(f"Timp mediu doar procesare:              {avg_proc_ms:.2f} ms")
    print(f"FPS end-to-end:                          {fps_end:.2f}")
    print(f"FPS doar procesare:                      {fps_proc:.2f}")

    # Afișăm ultimul cadru procesat în notebook
    img = Image.fromarray(last_frame)
    display(img)
else:
    print("Nu s-a procesat niciun cadru.")


In [ ]:
import cv2
import numpy as np
import time
from PIL import Image
from IPython.display import display
import pandas as pd
from pathlib import Path

# Parametri
FRAME_W = 640
FRAME_H = 480
NUM_FRAMES = 100
THRESH_VAL = 60
GAUSS_KERNEL = 5
GAUSS_SIGMA = 1.0

# Inițializăm camera
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Nu s-a putut deschide camera (index 0).")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_H)

total_end_to_end = 0.0
total_proc_only = 0.0
processed = 0
last_frame = None

# listă de log-uri pe cadru
log_rows = []

for i in range(NUM_FRAMES):
    t0 = time.time()
    ret, frame = cap.read()
    if not ret:
        print(f"Eroare la citirea cadrului {i}.")
        break

    frame = cv2.resize(frame, (FRAME_W, FRAME_H))

    # PIPELINE
    proc_start = time.time()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, thr = cv2.threshold(gray, THRESH_VAL, 255, cv2.THRESH_BINARY)
    gauss = cv2.GaussianBlur(thr, (GAUSS_KERNEL, GAUSS_KERNEL), GAUSS_SIGMA)

    sobelx = cv2.Sobel(gauss, cv2.CV_16S, 1, 0, ksize=3)
    sobely = cv2.Sobel(gauss, cv2.CV_16S, 0, 1, ksize=3)
    absx = cv2.convertScaleAbs(sobelx)
    absy = cv2.convertScaleAbs(sobely)
    sobel = cv2.addWeighted(absx, 0.5, absy, 0.5, 0)

    out_bgr = cv2.cvtColor(sobel, cv2.COLOR_GRAY2BGR)

    proc_end = time.time()

    # timpi
    t_end_to_end = proc_end - t0
    t_proc_only = proc_end - proc_start

    total_end_to_end += t_end_to_end
    total_proc_only += t_proc_only
    processed += 1
    last_frame = out_bgr

    # log pentru acest cadru
    log_rows.append({
        "platform": "RaspberryPi",   # modifici la nevoie: RaspberryPi4 / RaspberryPi5
        "frame_index": i,
        "t_capture_plus_proc_ms": t_end_to_end * 1000.0,
        "t_proc_ms": t_proc_only * 1000.0,
        "width": FRAME_W,
        "height": FRAME_H,
        "pipeline": "BGR->GRAY->THR->GAUSS->SOBEL->BGR"
    })

cap.release()

if processed > 0:
    avg_end_ms = (total_end_to_end / processed) * 1000.0
    avg_proc_ms = (total_proc_only / processed) * 1000.0
    fps_end = processed / total_end_to_end
    fps_proc = processed / total_proc_only

    print(f"Cadre procesate efectiv:                 {processed}")
    print(f"Timp mediu end-to-end (captură+proc):   {avg_end_ms:.2f} ms")
    print(f"Timp mediu doar procesare:              {avg_proc_ms:.2f} ms")
    print(f"FPS end-to-end:                          {fps_end:.2f}")
    print(f"FPS doar procesare:                      {fps_proc:.2f}")

    # Afișăm ultimul cadru procesat
    img = Image.fromarray(last_frame)
    display(img)

    # --- salvare CSV ---
    df_rpi = pd.DataFrame(log_rows)
    out_csv = Path.home() / "timing_rpi_sobel_pipeline.csv"
    df_rpi.to_csv(out_csv, index=False)
    print("Fișier CSV salvat la:", out_csv)

else:
    print("Nu s-a procesat niciun cadru.")
